In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_48/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob3") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "8")\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 18:34:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.table("spfix_dm.join_to_clickstream")

DataFrame[msisdn: int, competitor: string, host_name: string, request_cnt: int, request_days: int]

In [5]:
spark.table("spfix_dm.join_to_cnum")

DataFrame[msisdn: int, competitor: string, phone_num: string, call_cnt: int]

In [15]:
def build_competitors_contacts(spark, logical_date, logger):
    from pyspark.sql import functions as F

    business_month_start = datetime(logical_date.year, logical_date.month, 1).date()
    business_month_end = business_month_start + relativedelta(months=1, days= -1)

    logger.info('Processing sms part ...')
    sms_users = (
        spark.table("spfix_dm.join_to_cnum_sms")
        .withColumn('called_party_number', F.lower(F.regexp_replace(F.col('competitor'), '\.', '')))
        .groupby(['msisdn', 'called_party_number'])
        .agg(
            F.sum(F.col('call_sms')).alias('cnt_sms')
        )
    )

    logger.info('Processing calls part ...')

    calls_users = (
        spark.table("spfix_dm.join_to_cnum")
        .withColumnRenamed('phone_num', 'called_party_number')
        .withColumnRenamed('competitor', 'provider')
        .groupby(['msisdn', 'provider'])
        .agg(
            F.sum(F.col('call_cnt')).alias('cnt_calls')
        )
    )

    logger.info('Processing web part ...')
    web_users = (
        spark.table("spfix_dm.join_to_clickstream")
        .withColumnRenamed('host name', 'host')
        .withColumnRenamed('competitor', 'provider')
        
        .groupby(['msisdn', 'provider'])
        .agg(
            F.sum(F.col('request_cnt')).alias('cnt_req')
        )
    )

    final_df = (
        # join sms from competitors
        sms_users
        .withColumn('called_party_number', F.concat(F.lit('fix_comp_sms_cnt_'), F.col('called_party_number')))
        .groupBy('msisdn')
        .pivot('called_party_number')
        .agg(F.sum('cnt_sms'))
        
        .join(
            calls_users
            .withColumn('provider', F.lower(F.regexp_replace(F.col('provider'), '\.', '')))
            .withColumn('provider', F.concat(F.lit('fix_comp_calls_cnt_'), F.col('provider')))
            .groupBy('msisdn')
            .pivot('provider')
            .agg(F.sum('cnt_calls')),
            ['msisdn'], 'full'
        )
        
        .join(
            web_users
            .withColumn('provider', F.lower(F.regexp_replace(F.col('provider'), '\.', '')))
            .withColumn('provider', F.concat(F.lit('fix_comp_web_cnt_'), F.col('provider')))
            .groupBy('msisdn')
            .pivot('provider')
            .agg(F.sum('cnt_req')),
            ['msisdn'], 'full'
        )
        .fillna(0)
    )

    return final_df


In [16]:
log = logging.getLogger(__name__)
logical_date = datetime.now().date()

In [17]:
build_competitors_contacts(spark, logical_date, log)

DataFrame[msisdn: int, fix_comp_sms_cnt_akado: bigint, fix_comp_sms_cnt_beeline: bigint, fix_comp_sms_cnt_domru: bigint, fix_comp_sms_cnt_megafon: bigint, fix_comp_sms_cnt_rostelecom: bigint, fix_comp_calls_cnt_akado: bigint, fix_comp_calls_cnt_beeline: bigint, fix_comp_calls_cnt_domru: bigint, fix_comp_calls_cnt_megafon: bigint, fix_comp_calls_cnt_rostelecom: bigint, fix_comp_web_cnt_akado: bigint, fix_comp_web_cnt_beeline: bigint, fix_comp_web_cnt_domru: bigint, fix_comp_web_cnt_megafon: bigint, fix_comp_web_cnt_rostelecom: bigint]

In [7]:
build_competitors_contacts(spark, logical_date, log).count()

534691

In [18]:
df = build_competitors_contacts(spark, logical_date, log)

In [19]:
(
    df
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.mob_features_dm__3__base")
)

26/03/16 18:42:39 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [20]:
spark.stop()